# 56 — Fairness & bias measurement

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/056-fairness-and-bias-measurement.ipynb)

A single accuracy number can hide very different error rates. In this notebook you build the two group tables from the lesson, compute the gaps, and see why equal accuracy is not equal treatment.

## 1. Build the two group tables

Each group has 100 applicants: 40 truly qualified and 60 not. The four counts are TP, FP, FN and TN. Run the cell and read the rates row by row.

In [ ]:
import numpy as np                                    # arrays, preinstalled on Colab

counts = {                                            # group -> [TP, FP, FN, TN]
    "A": np.array([30, 20, 10, 40]),                  # 30 qualified approved, 20 not
    "B": np.array([20, 10, 20, 50]),                  # B loses 10 more qualified people
}
for name, (tp, fp, fn, tn) in counts.items():         # one group at a time
    n = tp + fp + fn + tn                             # group size = 100
    selection = (tp + fp) / n                         # approval rate
    tpr = tp / (tp + fn)                              # rate among qualified
    fpr = fp / (fp + tn)                              # rate among unqualified
    acc = (tp + tn) / n                               # overall accuracy
    print(f"{name}: selection={selection:.2f} TPR={tpr:.3f} FPR={fpr:.3f} acc={acc:.2f}")

## 2. Same accuracy, different errors

Both rows should print `acc=0.70`, yet A approves half its applicants and B approves 30%. The gaps below measure two different fairness claims.

In [ ]:
def gaps(table):                                      # dict group -> [TP, FP, FN, TN]
    sel, tpr = {}, {}                                 # one entry per group
    for name, (tp, fp, fn, tn) in table.items():      # unpack the four counts
        sel[name] = (tp + fp) / (tp + fp + fn + tn)   # selection rate
        tpr[name] = tp / (tp + fn)                    # true-positive rate
    dp = max(sel.values()) - min(sel.values())        # demographic parity gap
    eo = max(tpr.values()) - min(tpr.values())        # equal opportunity gap
    return dp, eo

dp, eo = gaps(counts)                                 # reuse the tables above
print(f"demographic parity gap = {dp:.2f}")           # expect 0.20
print(f"equal opportunity gap  = {eo:.2f}")           # expect 0.25
print("accuracy gap          =", 0.0)                # both are 70% correct

## Your turn

1. Change B's TP from 20 to 30 and TN from 50 to 40. Recompute both gaps — which one moves, and by how much?
2. Set B's selection rate equal to A's by moving ten unqualified B applicants from TN to FP. What happens to the equal-opportunity gap?
3. Explain in one sentence why the two definitions cannot both reach zero with a single global threshold when the groups have different base rates.